# Train balance bot using PPO with curriculum learning

Run each cell by pressing `shift + enter`.

In this notebook, we configure PPO and the balance bot environment (gymnasium wrapper for MuJoCo). We perform a simple 2-step curriculum learning. In the first phase, we train the bot to stay upright in any way possible. This will usually result in the bot driving around to stay up. In the second phase, we introduce a penalty for moving off the original location or rotating about the Z axis (yaw).

Note that we're using *privileged information* in the second phase. We rely on MuJoCo's `qpos` and `qvel` variables to get the robot's position and velocity information without using any built-in sensors. This helps shape the reward function, but it's not something we have easy access to on the real robot.

In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

# Import custom environment
from balance_bot_env import BalanceBotEnv

# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import PPO training module and exporter
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx

In [2]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 4               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

In [3]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 16,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 16,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [4]:
def make_balance_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), kw=kwargs: make_balance_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [5]:
# Phase 1: Balance only (don't worry about position or rotation)
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    position_penalty_coef=0.0,
    yaw_penalty_coef=0.0
)

# Choo choo train
result = train(ppo_config, envs=envs)

Run name: BalanceBot-v0__balance-bot-ppo__42__1788404013
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-ppo
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=6271.59) to runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/checkpoint_iter0100.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=9991.15) to runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/best_model.cleanrl_model


NaN or Inf found in input tensor.


Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9994.32) to runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/best_model.cleanrl_model
Final model saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/balance-bot-ppo_final.cleanrl_model
Best model mean return: 9994.32, saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/best_model.cleanrl_model


In [6]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/best_model.cleanrl_model
Final model: runs/BalanceBot-v0__balance-bot-ppo__42__1788404013/balance-bot-ppo_final.cleanrl_model
Best mean return: 9994.32
Loaded best model (mean_return=9994.32)


In [10]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

Mean return: 2038.09


In [11]:
# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Exported charts_metrics.csv (5 metrics, 526 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


In [9]:
# Phase 2: Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.position_penalty_coef = 0.001
    env.yaw_penalty_coef = 0.1

# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

Run name: BalanceBot-v0__balance-bot-ppo__42__1788409476
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-ppo
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=5909.96) to runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=9360.12) to runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=9952.05) to runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/checkpoint_iter0200.cleanrl_model
Final model saved to runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/balan

In [12]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/best_model.cleanrl_model
Final model: runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/balance-bot-ppo_final.cleanrl_model
Best mean return: 9952.05
Loaded best model (mean_return=9952.05)


In [13]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

Mean return: 2039.24


In [14]:
# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Exported charts_metrics.csv (5 metrics, 526 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


In [15]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3


In [16]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

/workspace/software/rl/ppo_trainer.py:381: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0903 14:57:45.181000 62875 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0903 14:57:45.184000 62875 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0903 14:57:45.189000 62875 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs/BalanceBot-v0__balance-bot-ppo__42__1788409476/actor.onnx


/opt/pyenv/versions/3.12.14/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
